# 📊 Flujo 5: Métricas y KPIs
### Lo que pide negocio

Totales, acumulados y
rankings. Aquí brillan
las Window functions.

In [ ]:
from pyspark.sql import (
    SparkSession, functions as F,
    Window)

spark = (
    SparkSession.builder
    .appName('metricas_kpis')
    .getOrCreate()
)

ventas = spark.createDataFrame([
  ('2025-01', 'Madrid', 100),
  ('2025-02', 'Madrid', 150),
  ('2025-03', 'Madrid', 120),
  ('2025-01', 'Sevilla', 80),
  ('2025-02', 'Sevilla', 200),
  ('2025-03', 'Sevilla', 90),
], ['mes','ciudad','importe'])

## 1️⃣ KPI simple
### Total por ciudad

In [ ]:
(ventas
 .groupBy('ciudad')
 .agg(F.sum('importe').alias('total'))
 .orderBy(F.col('total').desc())
 .show())

## 2️⃣ Acumulado mensual
### El 'running total'

Suma que crece mes a mes
en cada ciudad.

In [ ]:
w = (
  Window
  .partitionBy('ciudad')
  .orderBy('mes')
  .rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow)
)

(ventas
 .withColumn('acumulado',
   F.sum('importe').over(w))
 .orderBy('ciudad', 'mes')
 .show())

## 3️⃣ Variación mensual
### Con lag() miras atrás

Cada mes vs el anterior
de la misma ciudad.

In [ ]:
w2 = (
  Window
  .partitionBy('ciudad')
  .orderBy('mes')
)

(ventas
 .withColumn('mes_anterior',
   F.lag('importe').over(w2))
 .withColumn('variacion',
   F.col('importe')
   - F.col('mes_anterior'))
 .orderBy('ciudad', 'mes')
 .show())

## 4️⃣ Ranking
### Mejor mes de cada ciudad

In [ ]:
w3 = (
  Window
  .partitionBy('ciudad')
  .orderBy(F.col('importe').desc())
)

(ventas
 .withColumn('rank',
   F.rank().over(w3))
 .filter(F.col('rank') == 1)
 .select('ciudad', 'mes', 'importe')
 .show())

spark.stop()